In [1]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, GridSearchCV ,cross_validate
from sklearn.metrics import *
import joblib

In [3]:
train_data = pd.read_csv('data01_train.csv')
test_data = pd.read_csv('data01_test.csv')

In [4]:
train_data.drop('subject', axis=1, inplace=True)
test_data.drop('subject', axis=1, inplace=True)

In [5]:
dynamic_activity = (train_data['Activity']=='WALKING') | (train_data['Activity']=='WALKING_UPSTAIRS') | (train_data['Activity']=='WALKING_DOWNSTAIRS')
train_data = train_data[dynamic_activity]

In [6]:
dynamic_activity = (test_data['Activity']=='WALKING') | (test_data['Activity']=='WALKING_UPSTAIRS') | (test_data['Activity']=='WALKING_DOWNSTAIRS')
test_data = test_data[dynamic_activity]

In [7]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [8]:
X_train.shape

(2647, 561)

In [9]:
X_test = test_data.drop('Activity', axis=1)
y_test = test_data.loc[:, 'Activity']

In [10]:
y_train.unique()

array(['WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'], dtype=object)

In [11]:
y_test.unique()

array(['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS'], dtype=object)

In [12]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

In [13]:
y_test_encoded = le.transform(y_test)

In [14]:
le.classes_

array(['WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'], dtype=object)

## 학습 시작

### 1차원 데이터 형태

In [15]:
cnn = Sequential()

In [16]:
cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(561, 1)))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=6, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 559, 32)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 279, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 277, 32)        │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 138, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4416)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       565,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 569,382 (2.17 MB)

 Trainable params: 569,382 (2.17 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
cnn.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [19]:
cnn.fit(X_train, y_train_encoded, batch_size=32, epochs=25)

Epoch 1/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.6640 - loss: 0.8267
Epoch 2/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9594 - loss: 0.1374
Epoch 3/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9936 - loss: 0.0466
Epoch 4/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.9948 - loss: 0.0301
Epoch 5/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9943 - loss: 0.0211
Epoch 6/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9898 - loss: 0.0365
Epoch 7/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9892 - loss: 0.0260
Epoch 8/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9905 - loss: 0.0234
Epoch 9/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9967 - loss: 0.0085
Epoch 10/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 1.0000 - loss: 0.0028
Epoch 11/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 1.0000 - loss: 0.0010
Epoch 12/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy:

In [20]:
y_pred = cnn.predict(X_test)

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step


In [21]:
y_pred = np.argmax(y_pred, axis=1)

In [22]:
print('accuracy :',accuracy_score(y_test_encoded, y_pred))
print('='*60)
print(confusion_matrix(y_test_encoded, y_pred))
print('='*60)
print(classification_report(y_test_encoded, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test_encoded, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 194   1]
 [  0   0 215]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       228
           1       1.00      0.99      1.00       195
           2       1.00      1.00      1.00       215

    accuracy                           1.00       638
   macro avg       1.00      1.00      1.00       638
weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


## 2차원 데이터 형태

In [23]:
X_train = X_train.values

In [24]:
X_train

array([[ 0.28979493, -0.03553571, -0.15035406, ..., -0.69581913,
         0.28715354,  0.11138756],
       [ 0.39480719,  0.03409832,  0.09122925, ..., -0.70502855,
         0.26495154,  0.13775755],
       [ 0.33070828,  0.00756107, -0.06137116, ..., -0.55272916,
         0.25388462,  0.29125603],
       ...,
       [ 0.26496066, -0.03471921, -0.08814026, ..., -0.60160374,
         0.33599617,  0.15651767],
       [ 0.19156832,  0.01332843, -0.10517392, ..., -0.69027801,
         0.30319352, -0.04418751],
       [ 0.21278703, -0.04813008, -0.12100062, ..., -0.71800894,
         0.29285636,  0.0249202 ]])

In [25]:
X_train_reshaped = X_train.reshape(-1, 33, 17, 1)

In [26]:
X_train_reshaped.shape

(2647, 33, 17, 1)

In [27]:
cnn = Sequential()

In [28]:
cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(33, 17, 1)))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=6, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 33, 17, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 8, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 8, 32)      │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 141,542 (552.90 KB)

 Trainable params: 141,542 (552.90 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
cnn.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [31]:
cnn.fit(X_train_reshaped, y_train_encoded, batch_size=32, epochs=25)

Epoch 1/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.5142 - loss: 1.0648
Epoch 2/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.9143 - loss: 0.3247
Epoch 3/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.9493 - loss: 0.1494
Epoch 4/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9765 - loss: 0.0840
Epoch 5/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9828 - loss: 0.0609
Epoch 6/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9862 - loss: 0.0477
Epoch 7/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9939 - loss: 0.0284
Epoch 8/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9965 - loss: 0.0176
Epoch 9/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9974 - loss: 0.0129
Epoch 10/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9982 - loss: 0.0106
Epoch 11/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9984 - loss: 0.0100
Epoch 12/25
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy:

In [32]:
X_test.shape

(638, 561)

In [33]:
X_test = X_test.values

In [34]:
X_test_reshaped = X_test.reshape(-1, 33, 17, 1)

In [35]:
X_test_reshaped.shape

(638, 33, 17, 1)

In [36]:
y_pred = cnn.predict(X_test_reshaped)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


In [37]:
y_pred = np.argmax(y_pred, axis=1)

In [38]:
print('accuracy :',accuracy_score(y_test_encoded, y_pred))
print('='*60)
print(confusion_matrix(y_test_encoded, y_pred))
print('='*60)
print(classification_report(y_test_encoded, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test_encoded, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       228
           1       1.00      1.00      1.00       195
           2       1.00      1.00      1.00       215

    accuracy                           1.00       638
   macro avg       1.00      1.00      1.00       638
weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976
